# ContractRisk — juez 11: recuperación y diagnóstico de transporte

Runtime CPU. No repetir notebooks 07–10 ni Comparación 02. Este notebook no regenera respuestas contractuales; continúa la calibración sintética del juez. Conserva la nota 2 y su motivo de attribution_bad; reanuda attribution_good sin nota. Misma pregunta, modelo, prompt, rúbrica, semillas y configuración de generación.

Cambios técnicos versionados: espera máxima de lectura 180 segundos (antes 90); diagnóstico separado de HTTP, timeout, red y JSON de la API. Solo 503 sin nota admite UN reintento global, tras 60 segundos. No reintenta timeout/red/JSON, otros estados ni notas incorrectas. La reparación de formato original, como máximo una por juicio, no cambia. Si falla, conserva parciales y se detiene; no borrar registros para repetir.

Mismo proyecto gratuito y misma carpeta Drive: se verifican los seis asientos históricos y las cuatro marcas previas. Acumulado anterior USD 2,4706725 como estimaciones/reservas, no factura. El sublímite de esta recuperación es el mismo techo TOTAL ya autorizado USD 5, incluyendo todas las reservas. No son USD 5 adicionales, no se reinicia el registro, no se liberan reservas desconocidas y NO se activa facturación. Si falta margen o la vigencia de precios, se detiene.

Habilitar GEMINI_API_KEY y WANDB_API_KEY para este notebook, sin pegar valores. Confirmar GRATUITO solo si el proyecto sigue sin facturación. Usa CPU, no T4. Los mensajes completos del proveedor, claves y pensamiento privado no se guardan; solo códigos/categorías sanitizadas y la respuesta final pública del juez.

Si pasa el piloto, completa la regresión de 38 juicios (uno reutilizado, 37 nuevos). Aunque pase, la validación final y la confirmación de doce casos siguen pendientes de auditoría: no se inventa esa aprobación ni se califica el sistema final aquí. No garantiza que el proveedor haya recuperado disponibilidad.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_hydrogen_continuation_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '71912f1c5755790381a8e39c8a67801bcbf9243d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '7afdda0212bf38ad1accd043a968db722569c618'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'ece58275c2a19abfb6bbd2fe4b507731232e9083'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
confirmation = input('Confirma que el proyecto de GEMINI_API_KEY es Free Tier y NO tiene facturación activada. Escribe GRATUITO para continuar: ')
assert confirmation.strip() == 'GRATUITO', 'No se hará ninguna petición API sin confirmar el proyecto gratuito.'
del confirmation
print('Registro persistente de presupuesto en Drive. No borrar para repetir corridas.')


In [ ]:
RESULTS = ROOT / 'hydrogen_recovery11_results'
run_visible([sys.executable, '-u', 'final/run_hydrogen_recovery11.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-free-project', '--require-wandb'], 'hydrogen_recovery11.log')


## Interpretación

La nota anterior sigue intacta. judgements_total incluye el checkpoint; judgements_executed solo los nuevos juicios y generation_http_calls incluye intentos técnicos y formato. No mezclar esos denominadores. Un 503 recuperado sigue en el historial económico y de errores.

Si se detiene, guardar los resultados y no borrar registros de Drive. No activar pagos ni repetir el notebook por cuenta propia. La media de notas no certifica calidad: hay que superar todos los umbrales, auditar razones y ejecutar confirmación nueva revisada.


In [ ]:
RESULTS = ROOT / 'hydrogen_recovery11_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json', 'partial_regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for logfile in ROOT.glob('*.log'):
    shutil.copy2(logfile, RESULTS / logfile.name)
for name in ['hydrogen_calibration_config.json', 'hydrogen_budget_policy.json', 'api_budget_policy.json']:
    shutil.copy2(ROOT / 'final' / name, RESULTS / name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_hydrogen_recuperacion11_resultados'), 'zip', RESULTS)
files.download(package)
